# Module 7: Data Export

This module exports results in multiple formats (CSV, JSON) for sharing and further analysis.

In [ ]:
import streamlit as st
import pandas as pd
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add src to path
sys.path.append(os.path.join(os.path.dirname(''), '..', 'src'))

st.set_page_config(
    page_title="Module 7: Data Export",
    page_icon="📤",
    layout="wide",
    initial_sidebar_state="collapsed",
)

In [ ]:
# Load CSS
def local_css(file_name):
    with open(file_name) as f:
        st.markdown(f'<style>{f.read()}</style>', unsafe_allow_html=True)

local_css("streamlit_app/static/style.css")

In [ ]:
# Page header
st.markdown("""
<div style="text-align: center; margin-top: 0.5rem; margin-bottom: 1rem;">
    <div style="font-size: 3rem; line-height: 1;">📤</div>
    <h1 style="margin: 0.25rem 0 0 0; font-weight: 700; font-size: 1.5rem;">
        Module 7: Data Export
    </h1>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

In [ ]:
# Import module logic
from modules.module8_data_export import DataExportModule
from modules.integrity_tracker import IntegrityTracker

from workspace import current_user, get_store, normalize_user_id

In [ ]:
# Import module logic
from modules.module8_data_export import DataExportModule
from modules.integrity_tracker import IntegrityTracker

In [ ]:
# Load data from workspace
st.markdown("### Load Data")

user = current_user()
if user:
    store = get_store()
    uid = normalize_user_id(user)
    
    workspace_dir = os.path.join("data", "users", uid)
    
    if os.path.exists(workspace_dir):
        saved_files = [f for f in os.listdir(workspace_dir) if f.endswith('.csv')]
        
        if saved_files:
            selected_file = st.selectbox("Select saved dataset", saved_files)
            
            if st.button("📂 Load Dataset"):
                file_path = os.path.join(workspace_dir, selected_file)
                df = pd.read_csv(file_path)
                st.success(f"Loaded {len(df)} coordinates from {selected_file}")
                st.dataframe(df.head(10), use_container_width=True)
                st.session_state['loaded_data'] = df
        else:
            st.caption("No saved datasets found.")
    else:
        st.caption("No saved datasets found.")
else:
    st.caption("Sign in to load your saved datasets.")

In [ ]:
# Integrity check
if 'loaded_data' in st.session_state:
    df = st.session_state['loaded_data']
    
    tracker = IntegrityTracker()
    valid, error_msg = tracker.validate_input_for_module(df, 7)
    
    if not valid:
        st.error(error_msg)
        st.stop()

In [ ]:
# Export format selection
if 'loaded_data' in st.session_state:
    st.markdown("---")
    st.markdown("### Export Format")
    
    export_formats = st.multiselect(
        "Select export formats",
        ["csv", "json"],
        default=["csv", "json"]
    )

In [ ]:
# Run Module 7
if 'loaded_data' in st.session_state and export_formats:
    st.markdown("---")
    
    col1, col2 = st.columns([1, 3])
    with col1:
        run_module = st.button("▶️ Run Module 7", type="primary")
    with col2:
        st.caption("Export results in multiple formats")
    
    if run_module:
        module7 = DataExportModule()
        df, report = module7.export_data(st.session_state['loaded_data'], formats=export_formats)
        
        # Mark as Module 7 complete
        df = tracker.mark_module_complete(df, 7)
        
        # Display results
        summary = module7.get_success_summary()
        st.success(summary)
        
        st.subheader("Export Report")
        st.json(report)
        
        st.session_state['module7_results'] = df
        st.balloons()

In [ ]:
# Download exported files
if 'module7_results' in st.session_state:
    st.markdown("---")
    st.markdown("### Download Files")
    
    export_dir = "data/exports"
    if os.path.exists(export_dir):
        export_files = [f for f in os.listdir(export_dir) if f.endswith('.csv') or f.endswith('.json')]
        
        for file in export_files:
            file_path = os.path.join(export_dir, file)
            with open(file_path, 'rb') as f:
                st.download_button(
                    label=f"⬇️ Download {file}",
                    data=f.read(),
                    file_name=file,
                    mime="application/octet-stream"
                )

In [ ]:
# Navigation
st.markdown("---")
st.markdown("### Pipeline Complete")

st.success("🎉 Congratulations! You have completed the full ExoQ pipeline!")
st.caption("Your data is ready for analysis and sharing.")

if st.button("🏠 Return to Navigation Hub"):
    st.info("Returning to Navigation Hub...")